In [14]:
import os
import torch
from pathlib import Path
from tqdm.notebook import tqdm
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
# Paths
CHROMA_DB_PATH = "../data/chroma_db"
Path("../data/chroma_db").mkdir(parents=True, exist_ok=True)

# Settings
CHUNK_SIZE     = 512
CHUNK_OVERLAP  = 64
NUM_ARTICLES   = 5000     # start small — increase later
BATCH_SIZE     = 500      # store in batches to save RAM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"Articles to load : {NUM_ARTICLES}")

Device : cuda
Articles to load : 5000


In [3]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True}
)
print("Embedding model loaded on:", DEVICE)

# Quick test
test_embed = embeddings.embed_query("What is artificial intelligence?")
print("Embedding dimension:", len(test_embed))

Embedding model loaded on: cuda
Embedding dimension: 1024


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " "]
)

print("Streaming Wikipedia dataset...")
wiki = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",   # or latest available
    split="train",
    streaming=True
)
all_docs = []

for i, article in enumerate(tqdm(wiki, total=NUM_ARTICLES)):
    if i >= NUM_ARTICLES:
        break

    # Skip very short articles
    if len(article["text"]) < 200:
        continue

    chunks = splitter.create_documents(
        texts=[article["text"]],
        metadatas=[{
            "title"     : article["title"],
            "source"    : "wikipedia",
            "article_id": i
        }]
    )
    all_docs.extend(chunks)

print(f"Total chunks created : {len(all_docs)}")
print(f"Sample chunk :\n{all_docs[0].page_content[:300]}")

Streaming Wikipedia dataset...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

Total chunks created : 211190
Sample chunk :
Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state 


In [6]:
print("Storing chunks in ChromaDB...")

# Store in batches to avoid memory issues on 6GB GPU
vectorstore = None

for i in tqdm(range(0, len(all_docs), BATCH_SIZE)):
    batch = all_docs[i : i + BATCH_SIZE]

    if vectorstore is None:
        # First batch — create the store
        vectorstore = Chroma.from_documents(
            documents=batch,
            embedding=embeddings,
            persist_directory=CHROMA_DB_PATH
        )
    else:
        # Subsequent batches — add to existing store
        vectorstore.add_documents(batch)

print(f"Done! Total chunks in DB: {vectorstore._collection.count()}")

Storing chunks in ChromaDB...


  0%|          | 0/423 [00:00<?, ?it/s]

Done! Total chunks in DB: 211190


In [7]:
# Test that retrieval works
query = "What is machine learning?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"Result {i+1} — {doc.metadata['title']}")
    print(doc.page_content[:200])
    print("---")

Query: What is machine learning?

Result 1 — Artificial intelligence
There are several kinds of machine learning. Unsupervised learning analyzes a stream of data and finds patterns and makes predictions without any other guidance.
Supervised learning requires a human t
---
Result 2 — Artificial intelligence
Game theory describes rational behavior of multiple interacting agents, and is used in AI programs that make decisions that involve other agents.

Learning 
Machine learning is the study of programs t
---
Result 3 — Computer science
. Artificial intelligence and machine learning aim to synthesize goal-orientated processes such as problem-solving, decision-making, environmental adaptation, planning and learning found in humans and
---


In [5]:
from datasets import load_dataset

coqa = load_dataset("coqa", split="train")
print(f"CoQA conversations : {len(coqa)}")
print(f"Sample question    : {coqa[0]['questions'][0]}")
print(f"Sample answer      : {coqa[0]['answers']['input_text'][0]}")

import pandas as pd

coqa_df = pd.DataFrame({
    "question": [q for item in coqa for q in item["questions"]],
    "answer"  : [a for item in coqa for a in item["answers"]["input_text"]]
})

coqa_df.to_csv("../data/processed/coqa_eval.csv", index=False)
print(f"Saved {len(coqa_df)} Q&A pairs to coqa_eval.csv")

CoQA conversations : 7199
Sample question    : When was the Vat formally opened?
Sample answer      : It was formally established in 1475
Saved 108647 Q&A pairs to coqa_eval.csv


In [17]:
# Confirm everything is ready for Phase 3
print("=" * 50)
print("INGESTION COMPLETE — STATUS CHECK")
print("=" * 50)
print(f"Chunks in ChromaDB : {vectorstore._collection.count()}")
print(f"CoQA eval pairs    : {len(coqa_df)}")
print(f"Chroma DB path     : {CHROMA_DB_PATH}")
print(f"GPU memory used    : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory free    : {torch.cuda.memory_reserved()/1e9:.2f} GB")
print("\nReady for Phase 3 — Basic RAG chain!")

INGESTION COMPLETE — STATUS CHECK
Chunks in ChromaDB : 211190
CoQA eval pairs    : 108647
Chroma DB path     : ../data/chroma_db
GPU memory used    : 0.09 GB
GPU memory free    : 0.39 GB

Ready for Phase 3 — Basic RAG chain!
